# Prefect

This is a tutorial to use the Prefect cluster from Jupyter, without Dask.

In [1]:
import os
print(f"Internal Prefect server: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['PREFECT_PUBLIC']}/dashboard"
print(f"Public Prefect dashboard: {dashboard}")

Internal Prefect server: http://prefect-server:4200/api
Public Prefect dashboard: http://localhost:4200/dashboard


In [1]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
from resources.utils import *  # reload the global vars again

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000


In [4]:
%%bash
prefect block ls

          Blocks           
┏━━━━┳━━━━━━┳━━━━━━┳━━━━━━┓
┃ ID ┃ Type ┃ Name ┃ Slug ┃
┡━━━━╇━━━━━━╇━━━━━━╇━━━━━━┩
└────┴──────┴──────┴──────┘
  List Block Types using   
  `prefect block type ls`  


In [24]:
# Other imports
import logging
import os
import prefect
from resources.my_shared_utils import get_ip_address

# When deploying, the prefect flows and tasks must be implemented in a python module.
# We cannot implement them from jupyter cells.
import my_prefect

# Data to test the example flow
my_data = [
    "PrefectHQ/prefect",
    "pydantic/pydantic",
    "huggingface/transformers"
]

### Implement the `quickstart` tutorial
See: https://docs.prefect.io/v3/get-started/quickstart

When calling the flow as a normal python function, the flow and tasks are run by Prefect on your local client environment = your Jupyter or terminal.

This is the easiest way to test your Prefect code because the same environment, Python interpreter and files are shared between your client, flow and tasks. But this is less performant because your tasks are not distributed on the cluster.

In [ ]:
# Show my client IP address using a function from this local module
logging.warning(f"Client IP address: {get_ip_address()}")

# Run the flow
my_prefect.flow_show_stars(my_data)

<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs above that your client IP address is also used by the Prefect flow and tasks.
  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.

### Run flows in local processes
See: https://docs.prefect.io/v3/deploy/run-flows-in-local-processes

Create a deployment for a flow by calling the `serve` method.

As for the quickstart above, the same environment, Python interpreter and files are shared between your client, flow and tasks.

In [30]:
import asyncio
from fastapi.concurrency import run_in_threadpool

# Deploy the flow
task = my_prefect.hack_for_jupyter( # we need a hack to deploy from jupyter
    my_prefect.flow_show_stars.serve,
    name="serve-local-processes",
    tags=["tutorial"],
    parameters={"github_repos": my_data},
)
name = "flow-show-stars/serve-local-processes"
await my_prefect.wait_for_deployment(name)

Finished deploying prefect flow: 'flow-show-stars/serve-local-processes'
Your flow 'flow-show-stars' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'flow-show-stars/serve-local-processes'

You can also run your flow via the Prefect UI: http://prefect-server:4200/deployments/deployment/acb70a72-a90c-48ba-9acf-0f6dcdfc78ad



In [7]:
%%bash -s "$name"
# Trigger a run for this flow from the command line
prefect deployment run "$1"

Creating flow run for deployment 'flow-show-stars/serve-local-processes'...
Created flow run 'rational-dove'.
└── UUID: 08af62dc-ff17-417e-9892-00a01857ba03
└── Parameters: {'github_repos': ['PrefectHQ/prefect', 'pydantic/pydantic', 'huggingface/transformers']}
└── Job Variables: {}
└── Scheduled start time: 2025-01-24 12:13:55 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/08af62dc-ff17-417e-9892-00a01857ba03


In [10]:
from prefect.settings import PREFECT_UI_URL
print(f"""
########
# NOTE #
########

Don't use the internal domain from the logs above: {PREFECT_UI_URL.value()!r}, use the public domain instead: {os.environ['PREFECT_PUBLIC']}
""")


########
# NOTE #
########

Don't use the internal domain from the logs above: 'http://prefect-server:4200', use the public domain instead: http://localhost:4200



<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs above that your client IP address is also used by the Prefect flow and tasks.
  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.

### Deploy flows with Python

See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/deploy-via-python

Prefect offers a flexible way to deploy flows to dynamic infrastructure using the Python SDK. This approach allows you to target specific work pools and utilize dynamically provisioned infrastructure.

This is easier to deploy than with YAML (see next section) but less complete (e.g. cannot run additional scripts or pip install ...)

In [29]:
flow = await prefect.flow.from_source(
    source=".", #str(Path(__file__).parent),  # code stored in local directory
    entrypoint="my_prefect.py:flow_show_stars",
)
await flow.deploy(
    name="local-process-deploy-local-code",
    work_pool_name="my-prefect-pool",
    ignore_warnings=True,
)

Creating/updating deployments... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Successfully created/updated all deployments!

                              Deployments                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ Name                                            ┃ Status  ┃ Details ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩
│ flow-show-stars/local-process-deploy-local-code │ applied │         │
└─────────────────────────────────────────────────┴─────────┴─────────┘

To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'flow-show-stars/local-process-deploy-local-code'

You can also run your flow via the Prefect UI: http://prefect-server:4200/deployments/deployment/9723adf2-0f36-4bd6-87f2-b9e82c6bf351

UUID('9723adf2-0f36-4bd6-87f2-b9e82c6bf351')

In [31]:
!prefect deployment run 'flow-show-stars/local-process-deploy-local-code' --param github_repos="[\"tata\"]"

Creating flow run for deployment 
'flow-show-stars/local-process-deploy-local-code'...
Created flow run 'realistic-prawn'.
└── UUID: 02d0d4eb-2d6c-4825-b066-4ee20706e390
└── Parameters: {'github_repos': ['tata']}
└── Job Variables: {}
└── Scheduled start time: 2025-01-24 13:18:07 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/02d0d4eb-2d6c-4825-b066-4ee20706e390
